In [1]:
import MINGLE as mg
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import networkx as nx
import seaborn as sns
from matplotlib.cm import get_cmap
import numpy as np
import os
from scipy.spatial.distance import cdist

c:\Users\annet\anaconda3\envs\annette_MINGLE\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\annet\anaconda3\envs\annette_MINGLE\Lib\site-packages\muon\_core\preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):


In [2]:
file_path = r"Z:\MINGLE\Data\Intestine\05_25_HuBMAP_tunit.csv"
cells = mg.pp.read_file(file_path)

C:\Users\annet\Documents\Projects\MINGLE-Annette\MINGLE_scverse\MINGLE\src\MINGLE\pp\preprocessing.py:34: DtypeWarning: Columns (62,63,70) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)
c:\Users\annet\anaconda3\envs\annette_MINGLE\Lib\functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [7]:
prob_cols = [
'Innate Immune Enriched', 'Outer Follicle', 'Plasma Cell Enriched',
'Transit Amplifying Zone', 'Adaptive Immune Enriched', 'Stroma',
'Paneth Enriched', 'Smooth Muscle & Innate Immune', 'Mature Epithelial',
'Microvasculature', 'CD8+ T Enriched IEL', 'Stroma & Innate Immune',
'Macrovasculature', 'Innervated Stroma', 'Secretory Epithelial',
'Innervated Smooth Muscle', 'Smooth Muscle', 'Glandular Epithelial',
'CD66+ Mature Epithelial', 'Inner Follicle'
]

In [ ]:
# ==========================================================
# MULTI-LEVEL STORAGE STRUCTURE (CPU VERSION)
# ==========================================================

import networkx as nx

# Ensure AnnData
if not hasattr(cells, "obs"):
    raise TypeError("cells must be AnnData")

adata = cells

# ----------------------------------------------------------
# Initialize storage container
# ----------------------------------------------------------
if "levels" not in adata.uns:
    adata.uns["levels"] = {}

if "cpu" not in adata.uns["levels"]:
    adata.uns["levels"]["cpu"] = {}

# ----------------------------------------------------------
# Helper function
# ----------------------------------------------------------
def run_cpu_level(
    adata,
    level_name,
    cluster_col,
    neighborhood_col,
    prob_cols,
    threshold=0.25,
    top_n=25,
    region_key="unique_region"
):
    print(f"\nRunning {level_name.upper()} level (CPU)")

    # 1) Centroids
    centroids = mg.tl.centroid_Calculation(
        adata,
        cluster_col=cluster_col,
        neighborhood_col=neighborhood_col,
    )

    # 2) CPU GMM
    gmm_probs = mg.tl.cpu_gmm_probability(
        adata,
        centroids,
        cluster_col=cluster_col,
        neighborhood_col=neighborhood_col,
    )

    # Store probabilities
    adata.obsm[f"{level_name}_cpu_probabilities"] = gmm_probs.copy()

    # Store assignments
    adata.obs[f"{level_name}_cpu_assignment"] = gmm_probs.idxmax(axis=1)
    adata.obs[f"{level_name}_cpu_max_prob"] = gmm_probs.max(axis=1)

    # 3) Network graph
    G, top_pairs = mg.tl.build_neighborhood_pair_graph(
        adata,
        prob_cols,
        threshold=threshold,
        region_key=region_key,
        top_n=top_n,
    )

    graph_dict = nx.node_link_data(G)

    # 4) Store metadata
    adata.uns["levels"]["cpu"][level_name] = {
        "cluster_col": cluster_col,
        "neighborhood_col": neighborhood_col,
        "centroids": centroids,
        "top_pairs": top_pairs,
        "graph": graph_dict,
        "threshold": threshold,
        "top_n": top_n,
    }

    print(f"{level_name} CPU level stored successfully")


# ==========================================================
# RUN LEVELS (CPU)
# ==========================================================

run_cpu_level(
    adata,
    level_name="tissue",
    cluster_col="Community",
    neighborhood_col="Tissue Unit",
    prob_cols=prob_cols,
)

run_cpu_level(
    adata,
    level_name="community",
    cluster_col="Neighborhood",
    neighborhood_col="Community",
    prob_cols=prob_cols,
)

run_cpu_level(
    adata,
    level_name="neighborhood",
    cluster_col="Cell Type",
    neighborhood_col="Neighborhood",
    prob_cols=prob_cols,
)

print("\nAll CPU levels stored in adata.uns['levels']['cpu']")

In [5]:
# ==========================================================
# MULTI-LEVEL STORAGE STRUCTURE (GPU VERSION)
# ==========================================================

import networkx as nx

# Ensure AnnData
if not hasattr(cells, "obs"):
    raise TypeError("cells must be AnnData")

adata = cells

# ----------------------------------------------------------
# Initialize storage container
# ----------------------------------------------------------
if "levels" not in adata.uns:
    adata.uns["levels"] = {}

if "gpu" not in adata.uns["levels"]:
    adata.uns["levels"]["gpu"] = {}

# ----------------------------------------------------------
# Helper function
# ----------------------------------------------------------
def run_gpu_level(
    adata,
    level_name,
    cluster_col,
    neighborhood_col,
    prob_cols,
    threshold=0.25,
    top_n=25,
    region_key="unique_region"
):
    print(f"\nRunning {level_name.upper()} level (GPU)")

    # 1) Centroids
    centroids = mg.tl.centroid_Calculation(
        adata,
        cluster_col=cluster_col,
        neighborhood_col=neighborhood_col,
    )

    # 2) GPU GMM
    gmm_probs = mg.tl.gpu_gmm_probability(
        adata,
        centroids,
        cluster_col=cluster_col,
        neighborhood_col=neighborhood_col,
    )

    # Store probabilities
    adata.obsm[f"{level_name}_gpu_probabilities"] = gmm_probs.copy()

    # Store assignments
    adata.obs[f"{level_name}_gpu_assignment"] = gmm_probs.idxmax(axis=1)
    adata.obs[f"{level_name}_gpu_max_prob"] = gmm_probs.max(axis=1)

    # 3) Network graph
    G, top_pairs = mg.tl.build_neighborhood_pair_graph(
        adata,
        prob_cols,
        threshold=threshold,
        region_key=region_key,
        top_n=top_n,
    )

    graph_dict = nx.node_link_data(G)

    # 4) Store metadata
    adata.uns["levels"]["gpu"][level_name] = {
        "cluster_col": cluster_col,
        "neighborhood_col": neighborhood_col,
        "centroids": centroids,
        "top_pairs": top_pairs,
        "graph": graph_dict,
        "threshold": threshold,
        "top_n": top_n,
    }

    print(f"{level_name} GPU level stored successfully")


# ==========================================================
# RUN LEVELS (GPU)
# ==========================================================

run_gpu_level(
    adata,
    level_name="tissue",
    cluster_col="Community",
    neighborhood_col="Tissue Unit",
    prob_cols=prob_cols,
)

run_gpu_level(
    adata,
    level_name="community",
    cluster_col="Neighborhood",
    neighborhood_col="Community",
    prob_cols=prob_cols,
)

run_gpu_level(
    adata,
    level_name="neighborhood",
    cluster_col="Cell Type",
    neighborhood_col="Neighborhood",
    prob_cols=prob_cols,
)

print("\nAll GPU levels stored in adata.uns['levels']['gpu']")


Running TISSUE level (GPU)
GPU Processing batch 1/126...
GPU Processing batch 2/126...
GPU Processing batch 3/126...
GPU Processing batch 4/126...
GPU Processing batch 5/126...
GPU Processing batch 6/126...
GPU Processing batch 7/126...
GPU Processing batch 8/126...
GPU Processing batch 9/126...
GPU Processing batch 10/126...
GPU Processing batch 11/126...
GPU Processing batch 12/126...
GPU Processing batch 13/126...
GPU Processing batch 14/126...
GPU Processing batch 15/126...
GPU Processing batch 16/126...
GPU Processing batch 17/126...
GPU Processing batch 18/126...
GPU Processing batch 19/126...
GPU Processing batch 20/126...
GPU Processing batch 21/126...
GPU Processing batch 22/126...
GPU Processing batch 23/126...
GPU Processing batch 24/126...
GPU Processing batch 25/126...
GPU Processing batch 26/126...
GPU Processing batch 27/126...
GPU Processing batch 28/126...
GPU Processing batch 29/126...
GPU Processing batch 30/126...
GPU Processing batch 31/126...
GPU Processing batch

ValueError: Obsm 'tissue_gpu_probabilities' needs to be of one of <class 'numpy.ndarray'>, <class 'numpy.ma.MaskedArray'>, <class 'scipy.sparse._csr.csr_matrix'>, <class 'scipy.sparse._csc.csc_matrix'>, <class 'scipy.sparse._csr.csr_array'>, <class 'scipy.sparse._csc.csc_array'>, <class 'h5py._hl.dataset.Dataset'>, <class 'zarr.core.array.Array'>, <class 'anndata.compat.ZappyArray'>, <class 'anndata.abc.CSRDataset'>, <class 'anndata.abc.CSCDataset'>, <class 'anndata.compat.DaskArray'>, <class 'cupy.ndarray'>, <class 'cupyx.scipy.sparse._base.spmatrix'>, <class 'anndata.compat.AwkArray'>, or <class 'anndata.compat.XDataArray'>, not <class 'anndata._core.anndata.AnnData'>.

In [ ]:
# ==========================================================
# MULTI-LEVEL STORAGE STRUCTURE (GPU VERSION) - FIXED
# Handles mg.tl.gpu_gmm_probability returning AnnData
# ==========================================================

import numpy as np
import pandas as pd
import networkx as nx

# Ensure AnnData
if not hasattr(cells, "obs"):
    raise TypeError("cells must be AnnData")

adata = cells

# ----------------------------------------------------------
# Initialize storage container
# ----------------------------------------------------------
if "levels" not in adata.uns:
    adata.uns["levels"] = {}

if "gpu" not in adata.uns["levels"]:
    adata.uns["levels"]["gpu"] = {}

if "prob_cols" not in adata.uns:
    adata.uns["prob_cols"] = {}  # stores column labels for each stored matrix


# ----------------------------------------------------------
# Helper: coerce GPU GMM output -> (DataFrame probs, colnames)
# ----------------------------------------------------------
def _extract_prob_df(gmm_out, obs_names):
    """
    Returns a pandas DataFrame with index == obs_names and columns == probability labels.
    Supports:
      - pandas DataFrame
      - AnnData with probabilities in .obs[cols] OR .X (dense) with .var_names
      - numpy array (needs columns inferred upstream)
    """

    # Case 1: already a DataFrame
    if isinstance(gmm_out, pd.DataFrame):
        prob_df = gmm_out.copy()
        prob_df = prob_df.reindex(obs_names)
        return prob_df

    # Case 2: AnnData returned
    if hasattr(gmm_out, "obs") and hasattr(gmm_out, "obs_names"):
        # Try: probabilities stored in obs as numeric columns
        numeric_cols = gmm_out.obs.select_dtypes(include=[np.number]).columns.tolist()

        # If obs contains many numeric cols, we assume those are probs
        # (If you want stricter filtering, pass known prob_cols and use intersection.)
        if len(numeric_cols) > 0:
            prob_df = gmm_out.obs[numeric_cols].copy()
            prob_df = prob_df.reindex(obs_names)
            return prob_df

        # Otherwise try: probabilities in X with var_names
        if hasattr(gmm_out, "X") and gmm_out.X is not None:
            X = gmm_out.X
            if not isinstance(X, np.ndarray):
                X = X.toarray()
            cols = list(getattr(gmm_out, "var_names", [f"p{i}" for i in range(X.shape[1])]))
            prob_df = pd.DataFrame(X, index=gmm_out.obs_names, columns=cols).reindex(obs_names)
            return prob_df

        raise ValueError("Could not find probabilities in returned AnnData (.obs numeric cols or .X).")

    raise TypeError(f"Unsupported gmm_out type: {type(gmm_out)}")


# ----------------------------------------------------------
# GPU runner
# ----------------------------------------------------------
def run_gpu_level(
    adata,
    level_name,
    cluster_col,
    neighborhood_col,
    prob_cols_for_graph,
    threshold=0.25,
    top_n=25,
    region_key="unique_region"
):
    print(f"\nRunning {level_name.upper()} level (GPU)")

    # 1) Centroids
    centroids = mg.tl.centroid_Calculation(
        adata,
        cluster_col=cluster_col,
        neighborhood_col=neighborhood_col,
    )

    # 2) GPU GMM (may return AnnData)
    gmm_out = mg.tl.gpu_gmm_probability(
        adata,
        centroids,
        cluster_col=cluster_col,
        neighborhood_col=neighborhood_col,
    )

    # Extract probability table as DataFrame aligned to adata.obs_names
    prob_df = _extract_prob_df(gmm_out, adata.obs_names)

    # Store numeric matrix in obsm (AnnData requirement)
    key = f"{level_name}_gpu_probabilities"
    adata.obsm[key] = prob_df.to_numpy(dtype=np.float32)

    # Store column labels separately so you can reconstruct DataFrame later
    adata.uns["prob_cols"][key] = list(prob_df.columns)

    # Store assignment and max probability in obs
    adata.obs[f"{level_name}_gpu_assignment"] = prob_df.idxmax(axis=1).astype(str)
    adata.obs[f"{level_name}_gpu_max_prob"] = prob_df.max(axis=1).astype(float)

    # ----------------------------------------------------------
    # 3) Network graph (use actual GPU probability columns)
    # ----------------------------------------------------------

    adata_graph = adata.copy()

    # Reconstruct probability DataFrame
    prob_df_full = pd.DataFrame(
        adata.obsm[key],
        index=adata.obs_names,
        columns=adata.uns["prob_cols"][key]
    )

    # Inject probabilities into obs
    for col in prob_df_full.columns:
        adata_graph.obs[col] = prob_df_full[col]

    # IMPORTANT: use GPU column names, not external prob_cols
    actual_prob_cols = list(prob_df_full.columns)

    G, top_pairs = mg.tl.build_neighborhood_pair_graph(
        adata_graph,
        actual_prob_cols,  # <-- FIX HERE
        threshold=threshold,
        region_key=region_key,
        top_n=top_n,
    )

    graph_dict = nx.node_link_data(G)

    # 4) Store metadata
    adata.uns["levels"]["gpu"][level_name] = {
        "cluster_col": cluster_col,
        "neighborhood_col": neighborhood_col,
        "centroids": centroids,
        "top_pairs": top_pairs,
        "graph": graph_dict,
        "threshold": threshold,
        "top_n": top_n,
        "obsm_prob_key": key,
    }

    print(f"{level_name} GPU level stored successfully -> obsm['{key}']")


# ==========================================================
# RUN LEVELS (GPU)
# ==========================================================

run_gpu_level(
    adata,
    level_name="tissue",
    cluster_col="Community",
    neighborhood_col="Tissue Unit",
    prob_cols_for_graph=prob_cols,
)

run_gpu_level(
    adata,
    level_name="community",
    cluster_col="Neighborhood",
    neighborhood_col="Community",
    prob_cols_for_graph=prob_cols,
)

run_gpu_level(
    adata,
    level_name="neighborhood",
    cluster_col="Cell Type",
    neighborhood_col="Neighborhood",
    prob_cols_for_graph=prob_cols,
)

print("\nAll GPU levels stored in adata.uns['levels']['gpu']")


Running TISSUE level (GPU)
GPU Processing batch 1/126...
GPU Processing batch 2/126...
GPU Processing batch 3/126...
GPU Processing batch 4/126...
GPU Processing batch 5/126...
GPU Processing batch 6/126...
GPU Processing batch 7/126...
GPU Processing batch 8/126...
GPU Processing batch 9/126...
GPU Processing batch 10/126...
GPU Processing batch 11/126...
GPU Processing batch 12/126...
GPU Processing batch 13/126...
GPU Processing batch 14/126...
GPU Processing batch 15/126...
GPU Processing batch 16/126...
GPU Processing batch 17/126...
GPU Processing batch 18/126...
GPU Processing batch 19/126...
GPU Processing batch 20/126...
GPU Processing batch 21/126...
GPU Processing batch 22/126...
GPU Processing batch 23/126...
GPU Processing batch 24/126...
GPU Processing batch 25/126...
GPU Processing batch 26/126...
GPU Processing batch 27/126...
GPU Processing batch 28/126...
GPU Processing batch 29/126...
GPU Processing batch 30/126...
GPU Processing batch 31/126...
GPU Processing batch

KeyError: "Missing columns in cells.obs: ['Innate Immune Enriched', 'Outer Follicle', 'Plasma Cell Enriched', 'Transit Amplifying Zone', 'Adaptive Immune Enriched', 'Stroma', 'Paneth Enriched', 'Smooth Muscle & Innate Immune', 'Mature Epithelial', 'Microvasculature', 'CD8+ T Enriched IEL', 'Stroma & Innate Immune', 'Macrovasculature', 'Innervated Stroma', 'Secretory Epithelial', 'Innervated Smooth Muscle', 'Smooth Muscle', 'Glandular Epithelial', 'CD66+ Mature Epithelial', 'Inner Follicle']"

In [ ]:
def adata_catplot(
    cells,
    region='B010_Ileum',
    x_key='x',
    y_key='y',
    neigh_key='Neighborhood',
    region_key='unique_region',
    palette=None
):
    """
    Enhanced version:
    - Reads probabilities from .obsm if available
    - Uses adata.uns["prob_cols"] to reconstruct column names
    - Automatically works for CPU/GPU stored results
    """

    if not hasattr(cells, "obs"):
        raise TypeError("`cells` must be an AnnData object")

    adata = cells

    if neigh_key not in adata.obs.columns:
        raise KeyError(f"{neigh_key} not found in adata.obs")

    # ------------------------------------------------------
    # Determine probability matrix key automatically
    # ------------------------------------------------------
    # Example:
    # neigh_key = "tissue_gpu_assignment"
    # prob_key  = "tissue_gpu_probabilities"
    # ------------------------------------------------------

    if neigh_key.endswith("_assignment"):
        prob_key = neigh_key.replace("_assignment", "_probabilities")
    else:
        prob_key = None

    # ------------------------------------------------------
    # Subset region
    # ------------------------------------------------------
    adata_reg = adata[adata.obs[region_key] == region]

    assigned_labels = adata_reg.obs[neigh_key].astype(str)

    # ------------------------------------------------------
    # Extract probabilities from .obsm (if available)
    # ------------------------------------------------------
    assigned_probabilities = pd.Series(
        index=adata_reg.obs_names,
        data=np.nan
    )

    if prob_key is not None and prob_key in adata.obsm:

        # Reconstruct probability DataFrame
        if "prob_cols" not in adata.uns or prob_key not in adata.uns["prob_cols"]:
            raise KeyError(f"Column names for {prob_key} not found in adata.uns['prob_cols']")

        colnames = adata.uns["prob_cols"][prob_key]

        prob_df = pd.DataFrame(
            adata.obsm[prob_key],
            index=adata.obs_names,
            columns=colnames
        )

        prob_df_reg = prob_df.loc[adata_reg.obs_names]

        # Pull probability corresponding to assigned label
        for cell_id in adata_reg.obs_names:
            label = assigned_labels.loc[cell_id]
            if label in prob_df_reg.columns:
                assigned_probabilities.loc[cell_id] = prob_df_reg.loc[cell_id, label]

    # ------------------------------------------------------
    # Build visualization DataFrame
    # ------------------------------------------------------
    visualization_df = pd.DataFrame({
        'x': adata_reg.obs[x_key],
        'y': adata_reg.obs[y_key],
        'Assigned Neighborhood': assigned_labels,
        'Assigned Probability': assigned_probabilities,
        region_key: adata_reg.obs[region_key]
    }, index=adata_reg.obs_names)

    plt.rcParams["legend.markerscale"] = 15

    figs = adata_catplot(
        visualization_df,
        X='x',
        Y='y',
        exp=region_key,
        hue='Assigned Neighborhood',
        axis='off',
        invert_y=False,
        size=10,
        figsize=8,
        legend=True,
        legend_title_fontsize=18,
        legend_fontsize=12,
        title_fontsize=18,
        palette=palette if palette is not None else "tab10",
        dpi=150
    )

    return figs

In [ ]:
# ==========================================================
# COMPLETE MULTI-LEVEL VISUALIZATION SCRIPT
# Spatial + Raw Networks + Normalized GM Networks
# ==========================================================

import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.lines as mlines
from matplotlib.cm import get_cmap

# ----------------------------------------------------------
# USER SETTINGS
# ----------------------------------------------------------
analysis_type = "cpu"   # change to "gpu" if desired

levels_to_plot = ["tissue", "community", "neighborhood"]

region_key = "unique_region"

threshold = 0.25
top_n = 24

figsize = (12, 12)
dpi = 200

node_size = 2000
min_width_vis = 1.0
max_width_vis = 8.0
_eps = 1e-12
spring_k = 2
spring_seed = 28


# ==========================================================
# 1️⃣ SPATIAL ASSIGNMENT PLOTS
# ==========================================================

print("\n================ SPATIAL ASSIGNMENTS ================\n")

for level in levels_to_plot:

    assignment_key = f"{level}_{analysis_type}_assignment"

    if assignment_key not in adata.obs.columns:
        print(f"Missing {assignment_key}")
        continue

    print(f"\nPlotting {level.upper()} ({analysis_type.upper()})")

    adata_catplot(
        adata,
        neigh_key=assignment_key,
        region_key=region_key,
    )


# ==========================================================
# 2️⃣ RAW NETWORK GRAPHS (COUNT-BASED)
# ==========================================================

print("\n================ RAW NETWORK GRAPHS ================\n")

for level in levels_to_plot:

    print(f"\nPlotting RAW network: {level.upper()} ({analysis_type.upper()})")

    graph_dict = adata.uns["levels"][analysis_type][level]["graph"]
    G = nx.node_link_graph(graph_dict)

    if G.number_of_edges() == 0:
        print("Graph empty.")
        continue

    edge_weights = [G[u][v]["weight"] for u, v in G.edges()]

    pos = nx.spring_layout(G, k=spring_k, seed=spring_seed)

    plt.figure(figsize=figsize, dpi=dpi)

    nx.draw_networkx_nodes(G, pos, node_size=node_size)

    nx.draw_networkx_edges(
        G,
        pos,
        width=edge_weights,
        edge_color="gray"
    )

    nx.draw_networkx_labels(G, pos, font_size=12)

    plt.title(f"{level.upper()} RAW Network ({analysis_type.upper()})")
    plt.axis("off")
    plt.tight_layout()
    plt.show()


# ==========================================================
# 3️⃣ NORMALIZED GEOMETRIC-MEAN NETWORK GRAPHS
# ==========================================================

print("\n================ NORMALIZED GM NETWORKS ================\n")

for level in levels_to_plot:

    print(f"\nPlotting NORMALIZED GM network: {level.upper()} ({analysis_type.upper()})")

    prob_key = f"{level}_{analysis_type}_probabilities"

    if prob_key not in adata.obsm:
        print("Missing probability matrix.")
        continue

    probs = adata.obsm[prob_key]

    graph_dict = adata.uns["levels"][analysis_type][level]["graph"]
    G = nx.node_link_graph(graph_dict)

    if G.number_of_edges() == 0:
        print("Graph empty.")
        continue

    # ------------------------------------------
    # Compute totals per node
    # ------------------------------------------
    totals = (probs > threshold).sum(axis=0)
    totals_dict = totals.to_dict()

    edge_list = list(G.edges())
    enrichments = []

    for u, v in edge_list:
        cnt = G[u][v]["weight"]

        total_u = float(totals_dict.get(u, 0.0))
        total_v = float(totals_dict.get(v, 0.0))

        prop_u = (cnt / total_u) if total_u > 0 else 0.0
        prop_v = (cnt / total_v) if total_v > 0 else 0.0

        prop_u = max(prop_u, _eps)
        prop_v = max(prop_v, _eps)

        enr = np.sqrt(prop_u * prop_v)
        enrichments.append(enr)

    # ------------------------------------------
    # Normalize for visualization
    # ------------------------------------------
    e_min = min(enrichments)
    e_max = max(enrichments)

    if e_max == e_min:
        e_max = e_min + 1e-6

    norm_widths = [
        min_width_vis + (e - e_min) / (e_max - e_min) * (max_width_vis - min_width_vis)
        for e in enrichments
    ]

    pos = nx.spring_layout(G, k=spring_k, seed=spring_seed)

    plt.figure(figsize=figsize, dpi=dpi)

    nx.draw_networkx_nodes(G, pos, node_size=node_size)
    nx.draw_networkx_edges(G, pos, width=norm_widths, edge_color="gray")
    nx.draw_networkx_labels(G, pos, font_size=12)

    # ------------------------------------------
    # Legend for geometric mean
    # ------------------------------------------
    legend_levels = [e_min, (e_min + e_max) / 2.0, e_max]

    def to_width(e):
        return min_width_vis + (e - e_min) / (e_max - e_min) * (max_width_vis - min_width_vis)

    legend_handles = [
        mlines.Line2D([], [], color="gray", linewidth=to_width(e), label=f"{e:.2f}")
        for e in legend_levels
    ]

    plt.legend(
        handles=legend_handles,
        title="Geometric Mean\nProbability",
        frameon=False,
        loc="upper left"
    )

    plt.title(f"{level.upper()} NORMALIZED GM Network ({analysis_type.upper()})")
    plt.axis("off")
    plt.tight_layout()
    plt.show()